![logog](https://raw.githubusercontent.com/Pacific-AI-Corp/langtest/main/docs/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pacific-AI-Corp/langtest/blob/main/demo/tutorials/test-specific-notebooks/RandomOptions.ipynb)

**LangTest** is an open-source python library designed to help developers deliver safe and effective Natural Language Processing (NLP) models. Whether you are using **John Snow Labs, Hugging Face, Spacy** models or **OpenAI, Cohere, AI21, Hugging Face Inference API and Azure-OpenAI** based LLMs, it has got you covered. You can test any Named Entity Recognition (NER), Text Classification, fill-mask, Translation model using the library. We also support testing LLMS for Question-Answering, Summarization and text-generation tasks on benchmark datasets. The library supports 100+ out of the box tests. For a complete list of supported test categories, please refer to the [documentation](http://langtest.org/docs/pages/docs/test_categories).


# Getting started with LangTest

In [ ]:
%pip install "langtest[openai]==2.7.0"

# Harness and Its Parameters

The Harness class is a testing class for Natural Language Processing (NLP) and LLM models. It evaluates the performance of a NLP model on a given task using test data and generates a report with test results.Harness can be imported from the LangTest library in the following way.

In [26]:
#Import Harness from the LangTest library
from langtest import Harness

### Set environment for OpenAI

In [ ]:
import os 

os.environ['OPENAI_API_KEY'] = '<API KEY>'  # Add your OpenAI API key here
os.environ['OPENROUTER_API_KEY'] = '<API KEY>'

## Randomize Options Test For MCQ

In [28]:
prompt = """
You will answer **single-answer multiple-choice** questions.

**Do this:**

1. Read the question carefully (watch for words like **NOT**, **EXCEPT**, **BEST**, **MOST**).
2. Review **all** options A-E.
3. Choose the **one** best correct option.

**Output rules (STRICT):**

* Return **exactly one line** in the format: `LETTER. OPTION_VALUE`
* Keep the option text **exactly as given** (spelling/case/punctuation).
* **No** explanations, no extra text, no quotes, no code blocks, no trailing punctuation after the option text.
* If options include “None of the above” / “All of the above,” treat them like any other option.

**Template:**
Question:
{question}
Options:
{options}
Answer:

"""

In [29]:
from langtest.types import HarnessConfig

config : HarnessConfig = {
    "model_parameters": {
        "user_prompt": prompt,
    },
    "tests": {
        "defaults": {
            "min_pass_rate": 0.5,
        },
        "robustness": {
            "randomize_options": {
                "min_pass_rate": 0.1,             
            }
        }
    },
    "evaluation": {
        "metric": "llm_eval",
        "model": "gpt-4o",
        "hub": "openai",
    }
}

In [30]:
harness = Harness(
    task="question-answering",
    model={
        "model": "mistralai/mistral-medium-3.1",
        "hub": "openrouter",
        "type": "chat",
    },
    data={
        "data_source": "MedQA",
        "split": "test-tiny",
    },
    config=config
)

Test Configuration : 
 {
 "model_parameters": {
  "user_prompt": "\nYou will answer **single-answer multiple-choice** questions.\n\n**Do this:**\n\n1. Read the question carefully (watch for words like **NOT**, **EXCEPT**, **BEST**, **MOST**).\n2. Review **all** options A-E.\n3. Choose the **one** best correct option.\n\n**Output rules (STRICT):**\n\n* Return **exactly one line** in the format: `LETTER. OPTION_VALUE`\n* Keep the option text **exactly as given** (spelling/case/punctuation).\n* **No** explanations, no extra text, no quotes, no code blocks, no trailing punctuation after the option text.\n* If options include \u201cNone of the above\u201d / \u201cAll of the above,\u201d treat them like any other option.\n\n**Template:**\nQuestion:\n{question}\nOptions:\n{options}\nAnswer:\n\n"
 },
 "tests": {
  "defaults": {
   "min_pass_rate": 0.5
  },
  "robustness": {
   "randomize_options": {
    "min_pass_rate": 0.1
   }
  }
 },
 "evaluation": {
  "metric": "llm_eval",
  "model": "gpt-

### Generating the test cases.

In [32]:
harness.generate()

Generating testcases...:   0%|          | 0/1 [00:00<?, ?it/s]

In [33]:
testcases = harness.testcases()

In [34]:
testcases.head()

,category,test_type,original_question,options,perturbed_options
0,robustness,randomize_options,A junior orthopaedic surgery resident is compl...,A. Disclose the error to the patient but leave...,A. Report the physician to the ethics committe...
1,robustness,randomize_options,A 67-year-old man with transitional cell carci...,A. Inhibition of thymidine synthesis\nB. Inhib...,A. Inhibition of proteasome\nB. Inhibition of ...
2,robustness,randomize_options,Two weeks after undergoing an emergency cardia...,A. Renal papillary necrosis\nB. Allergic inter...,A. Eosinophilic granulomatosis with polyangiit...
3,robustness,randomize_options,A 39-year-old woman is brought to the emergenc...,"A. Coagulase-positive, gram-positive cocci for...","A. Encapsulated, gram-negative coccobacilli fo..."
4,robustness,randomize_options,A 35-year-old man comes to the physician becau...,A. Erythromycin ointment\nB. Ketotifen eye dro...,A. Latanoprost eye drops\nB. Fluorometholone e...


harness.generate() method automatically generates the test cases (based on the provided configuration)

### Running the tests

In [35]:
harness.run()

Running testcases... :   0%|          | 0/50 [00:00<?, ?it/s]

Called after harness.generate() and is to used to run all the tests.  Returns a pass/fail flag for each test.

### Generated Results

In [36]:
results = harness.generated_results()

In [39]:
results.head()

,category,test_type,original_question,options,perturbed_options,expected_result,actual_result,pass
0,robustness,randomize_options,A junior orthopaedic surgery resident is compl...,A. Disclose the error to the patient but leave...,A. Report the physician to the ethics committe...,B. Disclose the error to the patient and put i...,C. Disclose the error to the patient and put i...,True
1,robustness,randomize_options,A 67-year-old man with transitional cell carci...,A. Inhibition of thymidine synthesis\nB. Inhib...,A. Inhibition of proteasome\nB. Inhibition of ...,E. Cross-linking of DNA,D. Cross-linking of DNA,True
2,robustness,randomize_options,Two weeks after undergoing an emergency cardia...,A. Renal papillary necrosis\nB. Allergic inter...,A. Eosinophilic granulomatosis with polyangiit...,C. Cholesterol embolization,C. Cholesterol embolization,True
3,robustness,randomize_options,A 39-year-old woman is brought to the emergenc...,"A. Coagulase-positive, gram-positive cocci for...","A. Encapsulated, gram-negative coccobacilli fo...","B. Encapsulated, gram-negative coccobacilli fo...","A. Encapsulated, gram-negative coccobacilli fo...",True
4,robustness,randomize_options,A 35-year-old man comes to the physician becau...,A. Erythromycin ointment\nB. Ketotifen eye dro...,A. Latanoprost eye drops\nB. Fluorometholone e...,B. Ketotifen eye drops,E. Ketotifen eye drops,True


This method returns the generated results in the form of a pandas dataframe, which provides a convenient and easy-to-use format for working with the test results. You can use this method to quickly identify the test cases that failed and to determine where fixes are needed.

### Final Results

We can call `.report()` which summarizes the results giving information about pass and fail counts and overall test pass/fail flag.

In [38]:
harness.report()

,category,test_type,fail_count,pass_count,pass_rate,minimum_pass_rate,pass
0,robustness,randomize_options,12,38,76%,10%,True
